# microlearn — The Four Elements of Machine Learning

All ML models boil down to four learnable elements: **Points, Rules, Weights, and Distributions**.

This notebook runs the same binary classification task through all four, so you can see how each one stores patterns, optimises, and evaluates differently.

Based on Christoph Molnar's ["Points, Rules, Weights, Distributions"](https://mindfulmodeler.substack.com/p/points-rules-weights-distributions) article.

In [ ]:
import sys, time
sys.path.insert(0, '.')

from microlearn.core import (
    make_moons, make_blobs, make_circles,
    train_test_split, accuracy, precision, recall,
    plot_decision_boundary, plot_all
)
from microlearn.points import KNN
from microlearn.rules import DecisionTree
from microlearn.weights import LogisticRegression
from microlearn.distributions import NaiveBayes
from microlearn.hybrids import ModelTree

import matplotlib.pyplot as plt
%matplotlib inline

---
## Part 1: Same Data, Four Lenses

We generate a moons dataset and fit all four models. Same input, same task, four fundamentally different machines.

In [ ]:
# Generate data
X, y = make_moons(n_samples=300, noise=0.2, seed=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, seed=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

In [ ]:
# Fit all four elements
models = [
    KNN(k=5),
    DecisionTree(max_depth=5),
    LogisticRegression(lr=0.5, epochs=500),
    NaiveBayes(),
]

for m in models:
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    print(f"{type(m).__name__:25s}  accuracy={accuracy(y_test, preds):.3f}")

In [ ]:
# Plot decision boundaries side by side
titles = [
    "Points (KNN)",
    "Rules (Decision Tree)",
    "Weights (Logistic Reg.)",
    "Distributions (Naive Bayes)",
]
fig = plot_all(models, X_test, y_test, titles=titles)
fig.suptitle("Same Data, Four Learnable Elements", fontsize=15, fontweight="bold", y=1.02)
plt.show()

**Observation:** Same task, completely different decision boundaries. Each element has its own "language" for carving up the feature space.

---
## Part 2: What Gets Stored?

Each element stores "knowledge" in a radically different form.

In [ ]:
# POINTS: the model IS the training data
knn = models[0]
print(f"KNN model size: {knn.model_size()} stored points")
print(f"First 3 stored points: {knn.stored_points()[0][:3]}")
print(f"\n→ The model literally stores every training sample.")

In [ ]:
# RULES: the model is a set of if-then conditions
tree = models[1]
print(f"Decision tree: {tree.model_size()} rules\n")
tree.print_rules()
print(f"\n→ Human-readable if-then logic. Each rule tests one feature.")

In [ ]:
# WEIGHTS: the model is a weight vector + bias
lr = models[2]
w, b = lr.stored_weights()
print(f"Logistic regression: {lr.model_size()} parameters")
print(f"Weights: {[round(wi, 4) for wi in w]}")
print(f"Bias:    {round(b, 4)}")
print(f"\n→ The entire model fits in 3 numbers.")

In [ ]:
# DISTRIBUTIONS: the model is a set of distribution parameters
nb = models[3]
print(f"Naive Bayes: {nb.model_size()} parameters\n")
for c, params in nb.stored_distributions().items():
    print(f"Class {c}:")
    print(f"  means     = {[round(m, 4) for m in params['means']]}")
    print(f"  variances = {[round(v, 4) for v in params['variances']]}")
    print(f"  prior     = {round(params['prior'], 4)}")
print(f"\n→ The model stores Gaussian parameters per class, not data points or weights.")

---
## Part 3: How Optimisation Differs

Each element demands a fundamentally different optimisation strategy.

In [ ]:
# POINTS: fit is instant, predict is slow (lazy learning)
knn_new = KNN(k=5)

t0 = time.time()
knn_new.fit(X_train, y_train)
fit_time = time.time() - t0

t0 = time.time()
_ = knn_new.predict(X_test)
pred_time = time.time() - t0

print(f"KNN fit time:     {fit_time*1000:.2f} ms  ← just stores data")
print(f"KNN predict time: {pred_time*1000:.2f} ms  ← searches all stored points")
print(f"\n→ Optimisation happens at PREDICTION time, not training time.")

In [ ]:
# RULES: watch the greedy splitting with verbose=True
print("Decision tree greedy splitting (depth-limited to 3):\n")
tree_verbose = DecisionTree(max_depth=3, verbose=True)
tree_verbose.fit(X_train, y_train)
print(f"\n→ Each line is ONE greedy decision. The tree picks the single best")
print(f"  split, then recurses. This is how you survive combinatorial explosion.")

In [ ]:
# WEIGHTS: plot the loss curve (gradient descent)
lr_new = LogisticRegression(lr=0.5, epochs=500)
lr_new.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(lr_new.loss_history, color='#01696F', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title('Weights: Gradient Descent Loss Curve', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"→ Directly optimise a differentiable loss. The loss IS the eval metric.")

In [ ]:
# DISTRIBUTIONS: MLE is a closed-form solution
nb_new = NaiveBayes()

t0 = time.time()
nb_new.fit(X_train, y_train)
fit_time = time.time() - t0

print(f"Naive Bayes fit time: {fit_time*1000:.2f} ms")
print(f"\n→ No iteration, no gradients. MLE gives an exact solution:")
print(f"  mean = sample mean, variance = sample variance. Done.")

---
## Part 4: How Evaluation Differs

Standard metrics (accuracy, precision, recall) work for all models. But each representation also has its own natural evaluation criteria.

In [ ]:
# Shared metrics
print(f"{'Model':25s}  {'Accuracy':>10s}  {'Precision':>10s}  {'Recall':>10s}")
print("-" * 60)
for m, name in zip(models, titles):
    preds = m.predict(X_test)
    print(f"{name:25s}  {accuracy(y_test, preds):10.3f}  "
          f"{precision(y_test, preds):10.3f}  {recall(y_test, preds):10.3f}")

In [ ]:
# WEIGHTS: the training loss IS the evaluation metric
lr_model = models[2]
print(f"Logistic Regression final training loss: {lr_model.loss_history[-1]:.4f}")
print(f"→ Unique to weights: the same function used for optimisation IS the evaluation metric.")

In [ ]:
# DISTRIBUTIONS: log-likelihood is the natural evaluation metric
nb_model = models[3]
train_ll = nb_model.log_likelihood(X_train, y_train)
test_ll = nb_model.log_likelihood(X_test, y_test)
print(f"Naive Bayes log-likelihood (train): {train_ll:.2f}")
print(f"Naive Bayes log-likelihood (test):  {test_ll:.2f}")
print(f"→ Distribution-based models have their own evaluative logic: likelihoods.")

---
## Part 5: When Each Wins

Different data geometries favour different elements.

In [ ]:
datasets = {
    "Moons (complex boundary)": make_moons(300, noise=0.2, seed=42),
    "Blobs (linearly separable)": make_blobs(300, noise=0.4, seed=42),
    "Circles (concentric)": make_circles(300, noise=0.1, factor=0.5, seed=42),
}

print(f"{'Dataset':30s}  {'KNN':>8s}  {'Tree':>8s}  {'LogReg':>8s}  {'NB':>8s}")
print("-" * 70)

for name, (X_d, y_d) in datasets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, seed=42)
    results = []
    for Model, kwargs in [
        (KNN, {"k": 5}),
        (DecisionTree, {"max_depth": 5}),
        (LogisticRegression, {"lr": 0.5, "epochs": 500}),
        (NaiveBayes, {}),
    ]:
        m = Model(**kwargs)
        m.fit(X_tr, y_tr)
        acc = accuracy(y_te, m.predict(X_te))
        results.append(acc)
    
    best_idx = results.index(max(results))
    row = "  ".join(
        f"{r:8.3f}" + (" ★" if i == best_idx else "  ") 
        for i, r in enumerate(results)
    )
    print(f"{name:30s}  {row}")

print(f"\n→ No single element wins everywhere. The data geometry determines")
print(f"  which representation language naturally fits.")

---
## Part 6: The Hybrid (Rules + Weights)

The four elements are not silos. A **model-based tree** combines rules (for partitioning) with weights (for local decision boundaries).

In [ ]:
# Fit the hybrid alongside its parent elements
tree_only = DecisionTree(max_depth=2)
lr_only = LogisticRegression(lr=0.5, epochs=500)
hybrid = ModelTree(max_depth=2, min_samples_leaf=10)

for m in [tree_only, lr_only, hybrid]:
    m.fit(X_train, y_train)

compare_models = [tree_only, lr_only, hybrid]
compare_titles = [
    f"Rules only (acc={accuracy(y_test, tree_only.predict(X_test)):.3f})",
    f"Weights only (acc={accuracy(y_test, lr_only.predict(X_test)):.3f})",
    f"Rules + Weights (acc={accuracy(y_test, hybrid.predict(X_test)):.3f})",
]

fig = plot_all(compare_models, X_test, y_test, titles=compare_titles)
fig.suptitle("Hybrid: Model-based Tree (Rules + Weights)", fontsize=14, fontweight="bold", y=1.02)
plt.show()

print(f"→ The hybrid uses rules to partition the space, then fits")
print(f"  local logistic regressions at each leaf. Best of both worlds.")

---
## Summary

| Element | Stores | Optimises by | Evaluates with | Best for |
|---------|--------|-------------|----------------|----------|
| **Points** | Training instances | Nearest-neighbour search (at predict time) | Standard metrics | Low-dim data, local patterns |
| **Rules** | If-then conditions | Greedy recursive splitting | Coverage, accuracy, diversity | Tabular data, interpretability |
| **Weights** | Weight vector/tensor | Gradient descent on differentiable loss | Same loss as training | Unstructured data (images, text) |
| **Distributions** | Distribution parameters | Maximum likelihood estimation | Log-likelihood | Probabilistic reasoning |

Representation is a choice. The choice shapes everything downstream: how you store patterns, how you optimise, how you evaluate, and when the model wins or fails.